### Thử NER với qwen2.5 local llm
### File này sẽ load thử vài file txt của text data để xem qwen có nhận diện các trường trong output hay không

In [1]:
from langchain_ollama import ChatOllama
import json
from langchain_core.runnables import RunnablePassthrough
from langchain_community.document_loaders import TextLoader
from pathlib import Path
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


C:\Users\ACER\AppData\Local\Temp\ipykernel_29800\2520606264.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [4]:
# Load 1.txt từ test data.
data_path = Path("./data/test_data")

loader = TextLoader("../data/test_data/2.txt", encoding="utf-8")
docs = loader.load()
text = docs[0].page_content
print(f"doc: {text}")

doc: 1.  Tiền sử bệnh nội khoa
    Các bệnh lý mãn tính: nghi ngờ xơ gan do rượu
    Các yếu tố nguy cơ liên quan: Có tiền sử sử dụng rượu bia

2.  Tiền sử bệnh hiện tại
    Triệu chứng hiện tại: hội chứng não gan
    Các sự kiện trước khi nhập viện
    - đến bệnh viện  với hội chứng não gan
    - Bệnh nhân được chuyển viện trong tình trạng hội chứng não gan tiến triển, ý thức suy giảm
    - Bệnh nhân nhập viện vào ngày w tiến triển hội chứng não gan
    - không thể tự chăm sóc bản thân tại nhà

3.  Đánh giá tại bệnh viện
    Các thủ thuật đã thực hiện: đặt shunt dẫn lưu tĩnh mạch cửa qua da


In [5]:
template = """
Bạn là hệ thống trích xuất khái niệm y tế từ văn bản bệnh án tiếng Việt.

Nhiệm vụ của bạn:
Từ MỘT văn bản đầu vào duy nhất, hãy phát hiện các khái niệm y tế và phân loại chúng vào đúng type.

Chỉ được dùng các type sau:
1. TRIỆU_CHỨNG
2. TÊN_XÉT_NGHIỆM
3. KẾT_QUẢ_XÉT_NGHIỆM
4. CHẨN_ĐOÁN
5. THUỐC

Bạn chỉ được trả về JSON hợp lệ, không giải thích, không markdown, không ```json.

Output phải là một list các dictionary theo format:

[
  {{
    "text": "chuỗi y tế được trích xuất chính xác từ input",
    "type": "một trong 5 type hợp lệ",
    "position": [start, end]
  }}
]

Quy tắc bắt buộc:

- "text" phải là chuỗi xuất hiện nguyên văn trong input.
- "position" là vị trí ký tự của "text" trong input.
- Dùng quy ước Python slicing: input_text[start:end] == text.
- start là index ký tự bắt đầu.
- end là index ngay sau ký tự cuối cùng.
- Không tự sửa lỗi chính tả trong trường "text".
- Không tự dịch tiếng Việt sang tiếng Anh.
- Không thêm trường candidates.
- Không thêm trường assertions.
- Không thêm bất kỳ trường nào khác ngoài text, type, position.
- Nếu không phát hiện khái niệm y tế nào, trả về [].
- Không gộp tên xét nghiệm và kết quả xét nghiệm.
  Ví dụ: "WBC 14,43" phải tách thành:
  - "WBC" là TÊN_XÉT_NGHIỆM
  - "14,43" là KẾT_QUẢ_XÉT_NGHIỆM

Định nghĩa type:

1. THUỐC
Dùng cho tên thuốc hoặc cụm thuốc có liều lượng/cách dùng đi kèm.
Ví dụ:
- "amlodipine 10 mg po daily"
- "aspirin 81 mg po daily"
- "metoprolol succinate xl 50 mg po daily"
- "nystatin oral suspension 5 ml po qid:prn"

2. TRIỆU_CHỨNG
Dùng cho triệu chứng, biểu hiện lâm sàng, cảm giác khó chịu của bệnh nhân.
Ví dụ:
- "ho"
- "đau nhức"
- "sốt"
- "khó thở"
- "đau bụng"

3. CHẨN_ĐOÁN
Dùng cho tên bệnh, hội chứng, tình trạng bệnh lý đã được chẩn đoán.
Ví dụ:
- "tăng huyết áp"
- "đái tháo đường"
- "viêm phổi"
- "suy tim"

4. TÊN_XÉT_NGHIỆM
Dùng cho tên xét nghiệm, chỉ số xét nghiệm, marker, tên cận lâm sàng.
Ví dụ:
- "WBC"
- "NEUT%"
- "CRP"
- "creatinin"
- "glucose máu"

5. KẾT_QUẢ_XÉT_NGHIỆM
Dùng cho giá trị kết quả xét nghiệm hoặc mô tả kết quả xét nghiệm.
Ví dụ:
- "14,43"
- "76,4"
- "dương tính"
- "âm tính"
- "tăng"
- "giảm"

Ví dụ mẫu:

Input:
"Danh sách thuốc trước nhập viện chính xác và đầy đủ. 1. amlodipine 10 mg po daily 2. aspirin 81 mg po daily 3. metoprolol succinate xl 50 mg po daily 4. guaifenesin ml po q6h:prn điều trị ho 5. nystatin oral suspension 5 ml po qid:prn điều trị đau nhức..."

Output:
[
  {{
    "text": "amlodipine 10 mg po daily",
    "type": "THUỐC",
    "position": [58, 83]
  }},
  {{
    "text": "aspirin 81 mg po daily",
    "type": "THUỐC",
    "position": [89, 111]
  }},
  {{
    "text": "metoprolol succinate xl 50 mg po daily",
    "type": "THUỐC",
    "position": [115, 154]
  }},
  {{
    "text": "guaifenesin ml po q6h:prn",
    "type": "THUỐC",
    "position": [158, 184]
  }},
  {{
    "text": "ho",
    "type": "TRIỆU_CHỨNG",
    "position": [196, 198]
  }},
  {{
    "text": "nystatin oral suspension 5 ml po qid:prn",
    "type": "THUỐC",
    "position": [202, 242]
  }},
  {{
    "text": "đau nhức",
    "type": "TRIỆU_CHỨNG",
    "position": [254, 262]
  }}
]

Bây giờ hãy xử lý văn bản sau.

Input:
{input_text}

Output:
"""
qwen = ChatOllama(model="qwen2.5:7b", temperature=0)
embedding_prompt = ChatPromptTemplate.from_template(template)
generate_emd_json = (
     embedding_prompt
    | qwen
    | StrOutputParser()
)
json_result = generate_emd_json.invoke({
    "input_text": text
})
print(f"Result: {json_result}")


Result: [
  {
    "text": "xơ gan do rượu",
    "type": "CHẨN_ĐOÁN",
    "position": [39, 51]
  },
  {
    "text": "hội chứng não gan",
    "type": "CHẨN_ĐOÁN",
    "position": [86, 102],
    "position": [247, 263],
    "position": [295, 311]
  },
  {
    "text": "đặt shunt dẫn lưu tĩnh mạch cửa qua da",
    "type": "CHẨN_ĐOÁN",
    "position": [338, 370]
  }
]


### So với kết quả của claude sonnet 5 medium (cùng prompt và cùng input)
```json
[
  {
    "text": "xơ gan do rượu",
    "type": "CHẨN_ĐOÁN",
    "position": [61, 75]
  },
  {
    "text": "hội chứng não gan",
    "type": "CHẨN_ĐOÁN",
    "position": [191, 208]
  },
  {
    "text": "hội chứng não gan",
    "type": "CHẨN_ĐOÁN",
    "position": [270, 287]
  },
  {
    "text": "hội chứng não gan",
    "type": "CHẨN_ĐOÁN",
    "position": [338, 355]
  },
  {
    "text": "ý thức suy giảm",
    "type": "TRIỆU_CHỨNG",
    "position": [368, 383]
  },
  {
    "text": "hội chứng não gan",
    "type": "CHẨN_ĐOÁN",
    "position": [432, 449]
  }
]
```


### So với kết quả của chatgpt5.5 high (cùng prompt và cùng input)
```json
[
{
"text": "xơ gan do rượu",
"type": "CHẨN_ĐOÁN",
"position": [61, 75]
},
{
"text": "hội chứng não gan",
"type": "CHẨN_ĐOÁN",
"position": [191, 208]
},
{
"text": "hội chứng não gan",
"type": "CHẨN_ĐOÁN",
"position": [270, 287]
},
{
"text": "hội chứng não gan",
"type": "CHẨN_ĐOÁN",
"position": [338, 355]
},
{
"text": "ý thức suy giảm",
"type": "TRIỆU_CHỨNG",
"position": [368, 383]
},
{
"text": "hội chứng não gan",
"type": "CHẨN_ĐOÁN",
"position": [432, 449]
}
]
```


# Đánh giá
Đầu tiên, có thể thấy Claude và GPT đều phân loại trường text tốt hơn Qwen, trong khi Qwen bỏ sót khá nhiều. Tiếp theo, đối với trường position, kết quả của Claude và GPT trùng khớp với nhau, còn Qwen thì hầu như phân loại sai. Cuối cùng, ở trường type, cả ba mô hình đều cần được cải thiện thêm.

# Hướng phát triển
Thử cái model local pretrained cho y khoa. <br>
Fine tune model bằng custom dataset mới.